# Play MCTS Bot

Play the local ResNet-B checkpoint through a fixed-budget MCTS bot. The notebook uses 300 MCTS simulations per bot move. MCTS uses a simulation budget, not a fixed minimax depth.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
from IPython.display import display
import importlib
import chess

from mcchess.bots import MCTSBot
import mcchess.bots.notebook as notebook_widgets

notebook_widgets = importlib.reload(notebook_widgets)
ClickableChessBoard = notebook_widgets.ClickableChessBoard
NotebookChessGame = notebook_widgets.NotebookChessGame

In [ ]:
CHECKPOINT_PATH = project_root / "runs" / "lichess_2026_05_2000plus_resnet_b_epoch20_cached_batchmetrics" / "checkpoint.pt"
MCTS_SIMULATIONS = 300
C_PUCT = 1.5
INFERENCE_DEVICE = "auto"

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Missing checkpoint: {CHECKPOINT_PATH}")
if not 200 <= MCTS_SIMULATIONS <= 400:
    raise ValueError("MCTS_SIMULATIONS should stay between 200 and 400 for this notebook")

bot = MCTSBot.from_checkpoint(
    CHECKPOINT_PATH,
    device=INFERENCE_DEVICE,
    name=f"resnet_b_mcts_{MCTS_SIMULATIONS}",
    simulations=MCTS_SIMULATIONS,
    c_puct=C_PUCT,
)
metadata = bot.checkpoint.metadata
{
    "checkpoint": str(metadata.path),
    "epoch": metadata.epoch,
    "val_total_loss": metadata.metrics.get("val_total_loss"),
    "completed_at": metadata.completed_at,
    "device": str(bot.device),
    "mcts_simulations": bot.config.simulations,
    "c_puct": bot.config.c_puct,
}

In [ ]:
game = NotebookChessGame(bot, human_color=chess.WHITE)
ui = ClickableChessBoard(game)
display(ui.widget)